## Dealing with nulls in Spark Data Frames

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Dealing with nulls in Spark Data Frames").getOrCreate()

24/03/18 22:38:46 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/18 22:38:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/18 22:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/18 22:38:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/18 22:38:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/18 22:38:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/03/18 22:38:48 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [8]:
employees = [
    (1, 'Scott', None, 1000.0, '10',
         'United States', '+1 123 456 7890', '123 45 6789'),
    (2, 'Henry', 'Ford', 1250.0, None,
         'Italy', '+39 02 26 83 01 02.', '011 45 6789'),
    (3, 'Nick', 'Junior', 750.0, ' ',
         'United Kingdom', '+44 7975 556677', '090 56 9870'),
    (4, 'Bill', '', 1500.0, None,  
         'Australia', '+44 20 7183 8750', '119 56 9870'),
]

In [9]:
employees

[(1,
  'Scott',
  None,
  1000.0,
  '10',
  'United States',
  '+1 123 456 7890',
  '123 45 6789'),
 (2,
  'Henry',
  'Ford',
  1250.0,
  None,
  'Italy',
  '+39 02 26 83 01 02.',
  '011 45 6789'),
 (3,
  'Nick',
  'Junior',
  750.0,
  ' ',
  'United Kingdom',
  '+44 7975 556677',
  '090 56 9870'),
 (4, 'Bill', '', 1500.0, None, 'Australia', '+44 20 7183 8750', '119 56 9870')]

In [10]:
employeesDF = spark.createDataFrame(employees, schema='''
    employee_id INT, first_name STRING, last_name STRING, 
    salary DOUBLE, bonus STRING, nationality STRING, contact_no STRING, ssn STRING
    ''')

In [11]:
employeesDF.show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+



In [18]:
from pyspark.sql.functions import coalesce, col, lit

In [17]:
employeesDF.withColumn('bonus_upd', col('bonus').cast('int')) \
    .show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|bonus_upd|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|       10|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|     NULL|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|     NULL|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|     NULL|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+



In [21]:
employeesDF.withColumn('bonus_upd', coalesce(col('bonus').cast('int'), lit(0))) \
    .show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|bonus_upd|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|       10|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|        0|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|        0|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|        0|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+



In [22]:
from pyspark.sql.functions import expr, nvl

In [34]:
employeesDF.withColumn('bonus_upd', expr("nvl(nullif(bonus, ' '), 0)")) \
    .show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|bonus_upd|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|       10|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|        0|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|        0|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|        0|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+



In [33]:
employeesDF.withColumn('bonus_upd', expr("nullif(bonus, ' ')")) \
    .show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|bonus_upd|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|       10|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|     NULL|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|     NULL|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|     NULL|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+---------+



In [44]:
employeesDF \
    .withColumn('payment', col('salary') + col('salary') * coalesce(col('bonus').cast('INT')/lit(100), lit(0))) \
    .show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+-------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|payment|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+-------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789| 1100.0|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789| 1250.0|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|  750.0|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870| 1500.0|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+-------+



In [45]:
help(employeesDF.na)

Help on DataFrameNaFunctions in module pyspark.sql.dataframe object:

class DataFrameNaFunctions(builtins.object)
 |  DataFrameNaFunctions(df: pyspark.sql.dataframe.DataFrame)
 |  
 |  Functionality for working with missing data in :class:`DataFrame`.
 |  
 |  .. versionadded:: 1.4.0
 |  
 |  .. versionchanged:: 3.4.0
 |      Supports Spark Connect.
 |  
 |  Methods defined here:
 |  
 |  __init__(self, df: pyspark.sql.dataframe.DataFrame)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  drop(self, how: str = 'any', thresh: Union[int, NoneType] = None, subset: Union[str, Tuple[str, ...], List[str], NoneType] = None) -> pyspark.sql.dataframe.DataFrame
 |      Returns a new :class:`DataFrame` omitting rows with null values.
 |      :func:`DataFrame.dropna` and :func:`DataFrameNaFunctions.drop` are aliases of each other.
 |      
 |      .. versionadded:: 1.3.1
 |      
 |      .. versionchanged:: 3.4.0
 |          Supports Spark Connect.
 |      
 |      Pa

In [46]:
employeesDF.show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|          1|     Scott|     NULL|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|
|          4|      Bill|         |1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+



In [51]:
employeesDF.fillna('unknown', 'last_name').replace('', 'unknown').show()

+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|employee_id|first_name|last_name|salary|bonus|   nationality|         contact_no|        ssn|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+
|          1|     Scott|  unknown|1000.0|   10| United States|    +1 123 456 7890|123 45 6789|
|          2|     Henry|     Ford|1250.0| NULL|         Italy|+39 02 26 83 01 02.|011 45 6789|
|          3|      Nick|   Junior| 750.0|     |United Kingdom|    +44 7975 556677|090 56 9870|
|          4|      Bill|  unknown|1500.0| NULL|     Australia|   +44 20 7183 8750|119 56 9870|
+-----------+----------+---------+------+-----+--------------+-------------------+-----------+



In [53]:
employeesDF.dropna().show()

+-----------+----------+---------+------+-----+--------------+---------------+-----------+
|employee_id|first_name|last_name|salary|bonus|   nationality|     contact_no|        ssn|
+-----------+----------+---------+------+-----+--------------+---------------+-----------+
|          3|      Nick|   Junior| 750.0|     |United Kingdom|+44 7975 556677|090 56 9870|
+-----------+----------+---------+------+-----+--------------+---------------+-----------+

